In [ ]:
#pip install -U google-genai python-dotenv tqdm

# LLM-based Classification (glassdoor)

LLM-based classification of English Glassdoor reviews into Herzberg's 14 two-factor theory categories.

**Steps:**
1. Data cleaning: load raw CSV, drop empty rows, filter to English only, write to `reviews_clean`
2. Table setup: create `classifications_llm` schema
3. Classification: async Gemini batch inference, results written to `classifications_llm`

In [ ]:
from google import genai
from google.genai import types
import os
from dotenv import load_dotenv
import asyncio
import nest_asyncio
import sqlite3
import json
import random
from tqdm.asyncio import tqdm
import pandas as pd
from langdetect import detect, DetectorFactory

### Data Cleaning

Prepares the raw Glassdoor dataset: dropping all rows where "pros"/"cons" are empty, detecting language (only keeping english reviews)

In [ ]:
RAW_PATH = '/Users/franzi/Library/Mobile Documents/com~apple~CloudDocs/uni/employee_satisfaction_project/data/glassdoor_reviews.csv'
DB_PATH  = "PJDB.db"

# 1. Load
df = pd.read_csv(RAW_PATH)
print(f"Raw:  {len(df):,} rows")

# 2. Drop rows where both pros and cons are empty, and rows without a rating
df = df.dropna(subset=['pros', 'cons'], how='all')
df = df[(df['pros'] != "") | (df['cons'] != "")]
df = df.dropna(subset=['overall_rating'])
print(f"After dropping empty/no-rating rows: {len(df):,}")

# 3. Language detection — keep English only
DetectorFactory.seed = 0

def get_lang(text):
    try:
        return detect(str(text))
    except Exception:
        return 'unknown'

tqdm.pandas()
df['text_for_lang'] = df['pros'].fillna("") + " " + df['cons'].fillna("")
df['lang'] = df['text_for_lang'].progress_apply(get_lang)

print("\nLanguage distribution (top 12):")
print(df['lang'].value_counts().head(12).to_string())

df_en = df[df['lang'] == 'en'].copy()
print(f"\nEnglish reviews kept: {len(df_en):,}")

# 4. Clean column names and write to SQLite
df_en.columns = [c.replace('.', '_').replace(' ', '_').lower() for c in df_en.columns]

with sqlite3.connect(DB_PATH) as conn:
    df_en.to_sql('reviews_clean', conn, if_exists='replace', index=False)
    conn.execute('CREATE INDEX IF NOT EXISTS idx_rating ON reviews_clean(overall_rating)')

print("Done — written to reviews_clean in PJDB.db")

### Classification

Classifies a subset of Glassdoor reviews into Herzberg's 14 factors using Gemini.  
Each review's pros and cons are classified independently into 14 binary labels.  
Results are written to `classifications_llm` with one row per review.

In [ ]:
load_dotenv()
DB_NAME = "PJDB.db"

with sqlite3.connect(DB_NAME) as conn:
    conn.execute("""
        CREATE TABLE IF NOT EXISTS classifications_llm (
            review_id INTEGER PRIMARY KEY,
            p_company_policy_administration INTEGER,
            p_supervision INTEGER,
            p_interpersonal_relations INTEGER,
            p_working_conditions INTEGER,
            p_salary INTEGER,
            p_personal_life INTEGER,
            p_job_security INTEGER,
            p_achievement INTEGER,
            p_recognition INTEGER,
            p_work_itself INTEGER,
            p_responsibility INTEGER,
            p_advancement INTEGER,
            p_possibility_of_growth INTEGER,
            p_status INTEGER,
            c_company_policy_administration INTEGER,
            c_supervision INTEGER,
            c_interpersonal_relations INTEGER,
            c_working_conditions INTEGER,
            c_salary INTEGER,
            c_personal_life INTEGER,
            c_job_security INTEGER,
            c_achievement INTEGER,
            c_recognition INTEGER,
            c_work_itself INTEGER,
            c_responsibility INTEGER,
            c_advancement INTEGER,
            c_possibility_of_growth INTEGER,
            c_status INTEGER
        )
    """)

n_classified = conn.execute("SELECT COUNT(*) FROM classifications_llm").fetchone()[0]
n_total = conn.execute("SELECT COUNT(*) FROM reviews_clean").fetchone()[0]
print(f"classifications_llm: {n_classified:,} / {n_total:,} reviews classified")

In [ ]:
API_KEY = os.getenv("GEMINI_API_KEY")
BATCH_SIZE = 20
CONCURRENT_LIMIT = 30
MAX_RETRIES = 4

client = genai.Client(api_key=API_KEY)

FACTORS = [
    "company_policy_administration", "supervision", "interpersonal_relations",
    "working_conditions", "salary", "personal_life", "job_security",
    "achievement", "recognition", "work_itself", "responsibility",
    "advancement", "possibility_of_growth", "status",
]

SYSTEM_PROMPT = """
You are a precise text analyst specialized in organizational psychology.
Classify a list of employee reviews using Herzberg's 14 two-factor theory categories.

=== FACTOR DEFINITIONS ===

company_policy_administration: Company-level organisation, management policies, HR processes,
  bureaucracy, rules, communication structures, delegation of authority.
supervision: The direct supervisor's competence, fairness, behaviour, or willingness to
  delegate. Only applies to the immediate boss relationship.
interpersonal_relations: The social quality of relationships with colleagues, superiors, or
  subordinates — team atmosphere, collegiality, personal connections at work.
working_conditions: Physical environment (space, equipment, tools, noise, safety) and
  workload volume or intensity. Does NOT include work-life balance effects on private life.
salary: All forms of compensation: base pay, bonuses, benefits, raises.
personal_life: Work's impact on time and life OUTSIDE work — work-life balance, flexible
  hours, remote work, family-friendliness. Does NOT include workload volume itself.
job_security: Stability of the position, contract type, fear of layoffs, company financial
  health.
achievement: The employee's own sense of accomplishing something — completing a challenge,
  solving a problem, reaching a goal. Internal feeling of success.
recognition: External acknowledgement by others — praise, awards, feeling valued or ignored,
  criticism, blame. Requires another person doing the recognising.
work_itself: Intrinsic interest, variety, meaningfulness, or creativity of the actual tasks
  performed.
responsibility: Autonomy, ownership, freedom to make decisions, accountability for one's
  own work area.
advancement: Actual or potential movement up the career ladder — promotions, title changes,
  career progression within the company.
possibility_of_growth: Opportunities to learn and develop personally — training, new skills,
  courses, personal development programmes, access to education.
status: External prestige signals — company brand recognition, impressive job title, rank
  symbols, working for a well-known organisation.

=== DISAMBIGUATION RULES ===

These pairs are easily confused — use these rules to decide:

advancement vs possibility_of_growth:
  "I got promoted" or "no career path here" → advancement
  "I learned a lot" or "no training offered" → possibility_of_growth
  A review can trigger BOTH if it mentions both promotion AND learning.

supervision vs interpersonal_relations:
  Direct boss managing, evaluating, or delegating to the employee → supervision
  Team spirit, colleague relationships, social atmosphere → interpersonal_relations
  "My manager is great but the team is toxic" → supervision=1 AND interpersonal_relations=1

working_conditions vs personal_life:
  Heavy workload, bad equipment, noisy office → working_conditions
  "No time for family", flexible hours, work-life balance → personal_life
  "We work 60 hours a week in a noisy open-plan office" → BOTH can be 1

recognition vs achievement:
  "I feel proud of what I built" (internal feeling) → achievement
  "My boss never says thank you" (someone else acknowledging) → recognition
  "I solved a major bug and got praised for it" → BOTH can be 1

=== MULTI-LABEL RULE ===

A review text can — and often will — match multiple factors simultaneously.
Label ALL factors that are clearly present. Do not try to pick just one.

=== WHAT COUNTS AS A MENTION ===

Label a factor 1 for direct mentions AND clear unambiguous references where no
interpretation is needed. "My manager is completely useless" = supervision=1.
Do NOT label factors that require guessing or reading between the lines.
Detect irony and sarcasm and label the intended meaning accordingly.

=== INPUT / OUTPUT FORMAT ===

Input: a JSON array. Each element has "id", "pro" (positive aspects), and "con" (negative aspects).
Output: a single valid JSON object — no arrays, no markdown, no extra text.
  Keys = string review IDs.
  Each value has exactly "pro" and "con" keys.
  Each maps all 14 factor names to 0 or 1.

Example input:
[
  {"id": "1",
   "pro": "Fantastic salary and the team is like a family. I've grown a lot professionally.",
   "con": "My direct manager micromanages everything and HR processes are a nightmare."},
  {"id": "2",
   "pro": "The work itself is genuinely interesting and I have full ownership of my projects.",
   "con": "No promotions ever. Long hours bleed into my private life and no one says thank you."}
]

Example output:
{
  "1": {
    "pro": {"company_policy_administration": 0, "supervision": 0, "interpersonal_relations": 1,
            "working_conditions": 0, "salary": 1, "personal_life": 0, "job_security": 0,
            "achievement": 0, "recognition": 0, "work_itself": 0, "responsibility": 0,
            "advancement": 0, "possibility_of_growth": 1, "status": 0},
    "con": {"company_policy_administration": 1, "supervision": 1, "interpersonal_relations": 0,
            "working_conditions": 0, "salary": 0, "personal_life": 0, "job_security": 0,
            "achievement": 0, "recognition": 0, "work_itself": 0, "responsibility": 0,
            "advancement": 0, "possibility_of_growth": 0, "status": 0}
  },
  "2": {
    "pro": {"company_policy_administration": 0, "supervision": 0, "interpersonal_relations": 0,
            "working_conditions": 0, "salary": 0, "personal_life": 0, "job_security": 0,
            "achievement": 0, "recognition": 0, "work_itself": 1, "responsibility": 1,
            "advancement": 0, "possibility_of_growth": 0, "status": 0},
    "con": {"company_policy_administration": 0, "supervision": 0, "interpersonal_relations": 0,
            "working_conditions": 0, "salary": 0, "personal_life": 1, "job_security": 0,
            "achievement": 0, "recognition": 1, "work_itself": 0, "responsibility": 0,
            "advancement": 1, "possibility_of_growth": 0, "status": 0}
  }
}
"""

GENERATION_CONFIG = types.GenerateContentConfig(
    system_instruction=SYSTEM_PROMPT,
    response_mime_type="application/json",
    temperature=0,
)


def parse_batch_response(text):
    """Parse Gemini response, handling the three common malformed outputs."""
    try:
        result = json.loads(text)
    except json.JSONDecodeError:
        try:
            result, _ = json.JSONDecoder().raw_decode(text.strip())
        except json.JSONDecodeError as e:
            raise ValueError(f"JSON not parseable: {e}") from e

    if isinstance(result, list):
        normalized = {}
        for item in result:
            rid = str(item.get("id", ""))
            if not rid:
                continue
            normalized[rid] = {
                "pro": item.get("pro") or item.get("pros") or {},
                "con": item.get("con") or item.get("cons") or {},
            }
        result = normalized

    return result


def get_section(data, section):
    """Accept both 'pro'/'con' and 'pros'/'cons' as section keys."""
    alt = section + "s"
    return data.get(section) or data.get(alt) or {}


async def process_batch(semaphore, db_lock, rows, pbar):
    """Send one batch to Gemini and write results to DB, with retry logic."""
    async with semaphore:
        batch_input = [
            {"id": str(r[0]), "pro": str(r[1])[:2000], "con": str(r[2])[:2000]}
            for r in rows
        ]
        prompt = json.dumps(batch_input, ensure_ascii=False)

        for attempt in range(MAX_RETRIES):
            try:
                response = await client.aio.models.generate_content(
                    model="gemini-2.5-flash-lite",
                    contents=prompt,
                    config=GENERATION_CONFIG,
                )

                if not response.text:
                    raise ValueError("Empty response from model")

                batch_results = parse_batch_response(response.text)

                async with db_lock:
                    conn = sqlite3.connect(DB_NAME)
                    try:
                        cursor = conn.cursor()
                        for review_id, data in batch_results.items():
                            pro_data = get_section(data, "pro")
                            con_data = get_section(data, "con")

                            columns = ["review_id"]
                            placeholders = ["?"]
                            values = [int(review_id)]

                            for factor in FACTORS:
                                columns.append(f"p_{factor}")
                                placeholders.append("?")
                                values.append(int(pro_data.get(factor, 0)))

                            for factor in FACTORS:
                                columns.append(f"c_{factor}")
                                placeholders.append("?")
                                values.append(int(con_data.get(factor, 0)))

                            cursor.execute(
                                f"INSERT OR IGNORE INTO classifications_llm "
                                f"({', '.join(columns)}) VALUES ({', '.join(placeholders)})",
                                values,
                            )

                        conn.commit()
                    finally:
                        conn.close()

                pbar.update(len(rows))
                return

            except Exception as e:
                if attempt < MAX_RETRIES - 1:
                    wait = (2 ** attempt) + random.uniform(0, 1)
                    print(
                        f"\nAttempt {attempt + 1}/{MAX_RETRIES} failed "
                        f"(IDs {rows[0][0]}–{rows[-1][0]}): {e}. "
                        f"Retrying in {wait:.1f}s..."
                    )
                    await asyncio.sleep(wait)
                else:
                    print(
                        f"\nBatch permanently failed after {MAX_RETRIES} attempts "
                        f"(IDs {rows[0][0]}–{rows[-1][0]}): {e}"
                    )


async def main():
    with sqlite3.connect(DB_NAME) as conn:
        total_open = conn.execute(
            "SELECT COUNT(*) FROM reviews_clean "
            "WHERE rowid NOT IN (SELECT review_id FROM classifications_llm)"
        ).fetchone()[0]

    print(f"Reviews remaining: {total_open:,}")
    if total_open == 0:
        print("Nothing to do.")
        return

    semaphore = asyncio.Semaphore(CONCURRENT_LIMIT)
    db_lock = asyncio.Lock()
    pbar = tqdm(total=total_open, desc="Classifying")

    while True:
        with sqlite3.connect(DB_NAME) as conn:
            all_rows = conn.execute(
                """
                SELECT rowid, pros, cons
                FROM reviews_clean
                WHERE rowid NOT IN (SELECT review_id FROM classifications_llm)
                LIMIT ?
                """,
                (BATCH_SIZE * CONCURRENT_LIMIT,),
            ).fetchall()

        if not all_rows:
            break

        batches = [all_rows[i:i + BATCH_SIZE] for i in range(0, len(all_rows), BATCH_SIZE)]
        await asyncio.gather(*[process_batch(semaphore, db_lock, b, pbar) for b in batches])
        await asyncio.sleep(0.2)

    pbar.close()
    print("Classification complete.")


nest_asyncio.apply()
asyncio.run(main())

### Preview

In [ ]:
# Preview — sample output from classifications_llm
with sqlite3.connect(DB_NAME) as conn:
    preview = pd.read_sql(
        "SELECT * FROM classifications_llm ORDER BY review_id LIMIT 5", conn
    )

display(preview)